# Project 2 — Scanned Document Classification

This notebook provides a starting point for the **Scanned Document Classification** project. We classify eight kinds of scanned business documents using text extracted by OCR. The baseline intentionally uses only **TF-IDF + Logistic Regression**; the original page images remain available for later image-based or multimodal experiments.

The retained classes are advertisement, email, form, invoice, letter, news article, presentation, and resume. Use the validation split for development and leave the test split untouched until final evaluation.

## 1. Your Tasks

The provided notebook establishes a simple baseline for detecting potentially toxic comments. Your task is to **evaluate this baseline, identify its limitations, and investigate possible improvements**.

### 1.1 Explore the Dataset

Get familiar with the data and the classification task.

- Examine the dataset characteristics, class distribution, and representative examples.
- Consider how the target labels are defined and whether the data is suitable for the intended application.
- Identify potential issues related to data quality, coverage, or representativeness.

### 1.2 Evaluate the Baseline

Determine how well the model performs and which evaluation criteria are appropriate.

- Select and justify metrics relevant to the content moderation task.
- Evaluate overall and class-wise performance.
- Consider the consequences of different types of errors, particularly false positives and false negatives.

### 1.3 Analyze Errors and Diagnose Limitations

Investigate where and why the classifier makes mistakes.

- Examine misclassified examples and identify recurring patterns.
- Formulate possible explanations for the observed errors.
- Distinguish observations supported by the data from hypotheses that require further investigation.

### 1.4 Investigate Possible Improvements

Based on your observations and relevant literature, identify promising directions for improving the baseline.

- Research alternative approaches, representations, or model configurations.
- Explain which limitations these approaches might address.
- Where feasible, implement and evaluate an improvement against the original baseline using a consistent evaluation setup.

### 1.5 Document Your Findings

Summarize your experiments and the reasoning behind your decisions.

- What did you investigate, and why?
- What evidence supports your observations and conclusions?
- Which improvements appear promising, and what remains unresolved?

**Important:** Your decisions should be grounded in **empirical evidence or relevant research**, rather than assumptions. When proposing changes, explain the motivation and evaluate their effects whenever possible.

Use the validation set for model development and comparisons. Keep the test set reserved for final evaluation.



## 2. Starter code 

### 2.1 Prepare the dataset

Run the preparation script once from the repository root before opening this notebook. It reads the already-downloaded local dataset, samples reproducibly, copies selected images to a separate project directory, and caches Tesseract OCR. It never changes the source dataset.

```bash
sudo apt-get install tesseract-ocr tesseract-ocr-eng
python -m pip install pytesseract

python scripts/download_dataset.py "abarsky/rvl_cdip_small" "data/rvl_cdip_small"

python project_baselines/project_02/scripts/prepare_rvl_cdip_small.py \
  --examples-per-class 50 \
  --seed 42
```

By default, the script reads `data/rvl_cdip_small` and writes `project_baselines/project_02/data/prepared/rvl_cdip_small`. See `README.md` and `python scripts/prepare_rvl_cdip_small.py --help` for overrides.

Notice that above we only have 50 examples per class, which leaves only ~5 examples for validation. Increase that number to test with more documents.

### 2.2 Load and inspect the prepared data

Now that the data is local, you can load the dataset. 

In [ ]:
import os
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

# This works when Jupyter starts either in this project folder or at repository root.
project_dir = Path.cwd()
if not (project_dir / "scripts" / "prepare_rvl_cdip_small.py").exists():
    project_dir = project_dir / "project_baselines" / "project_02"
if not (project_dir / "scripts" / "prepare_rvl_cdip_small.py").exists():
    raise FileNotFoundError("Could not locate the project_02 directory.")

# RVL_CDIP_PREPARED_DIR is useful for temporary or shared data locations.
data_dir = Path(
    os.environ.get(
        "RVL_CDIP_PREPARED_DIR",
        project_dir / "data" / "prepared" / "rvl_cdip_small",
    )
).expanduser().resolve()
manifest_path = data_dir / "dataset.csv"
if not manifest_path.exists():
    raise FileNotFoundError(
        f"Prepared manifest not found at {manifest_path}. Run the preparation command above first."
    )

documents = pd.read_csv(manifest_path).fillna({"ocr_text": ""})
required_columns = {"image_path", "ocr_text", "label", "source_split"}
if not required_columns.issubset(documents.columns):
    raise ValueError(f"Manifest is missing: {required_columns - set(documents.columns)}")

print(f"Loaded {len(documents):,} documents from {data_dir}")
display(pd.crosstab(documents["label"], documents["source_split"]))

### 2.3 Images and OCR examples

The OCR text is an imperfect representation of each page. Compare it with the image before drawing conclusions about model errors.

In [ ]:
examples = (
    documents.groupby("label", group_keys=False)
    .sample(n=1, random_state=42)
    .sample(n=3, random_state=42)
)

fig, axes = plt.subplots(len(examples), 2, figsize=(13, 4 * len(examples)))
for row_number, (_, row) in enumerate(examples.iterrows()):
    image = Image.open(data_dir / row["image_path"])
    axes[row_number, 0].imshow(image, cmap="gray")
    axes[row_number, 0].set_title(f"{row['label']} ({row['source_split']})")
    axes[row_number, 0].axis("off")

    text = row["ocr_text"] or "[No OCR text extracted]"
    axes[row_number, 1].text(
        0, 1, textwrap.fill(text[:1_200], width=75), va="top", family="monospace"
    )
    axes[row_number, 1].set_title("Extracted OCR text (truncated)")
    axes[row_number, 1].axis("off")
plt.tight_layout()

### 2.4 Establishing a Baseline Classifier 

We prepare a TF-IDF + Logistic Regression baseline. We train only on the original training partition and evaluate on validation. Empty OCR is retained as an empty string rather than silently dropping difficult pages. The test partition is not used below.

In [ ]:
train_df = documents[documents["source_split"] == "train"].copy()
validation_df = documents[documents["source_split"] == "validation"].copy()
test_df = documents[documents["source_split"] == "test"].copy()

if train_df.empty or validation_df.empty:
    raise ValueError("The prepared sample must contain train and validation examples.")

print(f"Train: {len(train_df)}, validation: {len(validation_df)}, reserved test: {len(test_df)}")
print(f"Empty OCR — train: {(train_df['ocr_text'].str.len() == 0).sum()}, validation: {(validation_df['ocr_text'].str.len() == 0).sum()}")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=20_000,
    sublinear_tf=True,
)
X_train = vectorizer.fit_transform(train_df["ocr_text"])
X_validation = vectorizer.transform(validation_df["ocr_text"])

model = LogisticRegression(
    class_weight="balanced",
    max_iter=1_000,
    random_state=RANDOM_STATE,
)
model.fit(X_train, train_df["label"])
validation_predictions = model.predict(X_validation)
validation_probabilities = model.predict_proba(X_validation)

print("Training completed.")
print("TF-IDF representation:", X_train.shape)

### 2.5 Initial evaluation

Accuracy summarizes overall correctness. Macro-F1 gives every class equal weight and is useful even if a later sample is not perfectly balanced. The class-wise report and confusion matrix show where the baseline struggles.

In [ ]:
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
)

accuracy = accuracy_score(validation_df["label"], validation_predictions)
macro_f1 = f1_score(validation_df["label"], validation_predictions, average="macro")
print(f"Validation accuracy: {accuracy:.3f}")
print(f"Validation macro-F1: {macro_f1:.3f}\n")
print(
    classification_report(
        validation_df["label"], validation_predictions, zero_division=0
    )
)

fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay.from_predictions(
    validation_df["label"],
    validation_predictions,
    xticks_rotation=45,
    cmap="Blues",
    ax=ax,
)
ax.set_title("Validation confusion matrix")
plt.tight_layout()

### 5.1 A small error table

This table is only a starting point. Inspect the corresponding images and ask whether each failure comes from OCR, limited training data, overlapping document types, or the text model itself.

In [ ]:
validation_results = validation_df[["sample_id", "label", "image_path", "ocr_text"]].copy()
validation_results["prediction"] = validation_predictions
validation_results["confidence"] = validation_probabilities.max(axis=1)
errors = validation_results[
    validation_results["label"] != validation_results["prediction"]
].sort_values("confidence", ascending=False)
display(errors[["label", "prediction", "confidence", "ocr_text"]].head(10))

## 3. Your experiments